# 04 — Run Exp 2: Eigenspace rotation injection

Runs `ANALYSIS/perturbation/pe_stability_exp2.py`. Without touching the graph,
rotates eigenvectors inside near- and exactly-degenerate eigenpairs and
measures how much each model's scalar prediction moves across 20 random
SO(2) rotations.

**Prerequisites**
1. `MyDrive/datasets/pcqm4m-v2/processed/` contains the OGB-processed PCQM4Mv2 `.pt` file (run `00_setup.ipynb` once).
2. `MyDrive/stability_baselines/<method>/<seedX>/` contains `config.yaml` + a `*.ckpt` for each trained model you want to evaluate.

**Drive account:** **`znidar.mark@gmail.com`** (authorize in the Drive popup).

**Time budget on A100, 5000 graphs × 20 rotations per checkpoint:** ≈30–40 min / seed. With 7 checkpoints total that's ~3.5–5 h. Use `N_GRAPHS = 500` first to sanity-check. The one-time candidate scan (near-degenerate eigenpair search over the whole test set) takes a few minutes on CPU and is cached to `exp2_results/candidates.npz`.

## 1. Configuration — edit paths and seeds

In [ ]:
# =========================================================
# EDIT: which checkpoints to evaluate
# =========================================================
METHOD_RUN_DIRS = {
    "LapPE":            ["stability_baselines/lappe/seed1"],
    "SignNet-MLP":      ["stability_baselines/signnet_mlp/seed1"],
    "SignNet-DeepSets": ["stability_baselines/signnet_ds/seed1"],
    "L-HKS":            ["results_pcqm4m_subset/mlp_ablation/mlp3/seed2"],
    "fix-L-HKS":        ["results_pcqm4m_subset/mlp_ablation/mlp3_fixed/seed5"],
}

# EDIT: smaller number for fast sanity check (e.g. 500), 5000 for final figure.
N_GRAPHS   = 5000
BATCH_SIZE = 64

GITHUB_REPO     = "https://github.com/mark-znidar/gdl__2.git"
DRIVE_WORKSPACE = "/content/drive/MyDrive"
# =========================================================

DRIVE_DATASETS     = f"{DRIVE_WORKSPACE}/datasets"
DRIVE_RESULTS      = f"{DRIVE_WORKSPACE}/results_pcqm4m_subset"
DRIVE_STABILITY    = f"{DRIVE_WORKSPACE}/stability_baselines"
DRIVE_EXP_RESULTS  = f"{DRIVE_WORKSPACE}/pe_stability_results/exp2"

import os
os.makedirs(DRIVE_STABILITY,   exist_ok=True)
os.makedirs(DRIVE_EXP_RESULTS, exist_ok=True)

print("Methods to evaluate:")
for m, dirs in METHOD_RUN_DIRS.items():
    for d in dirs:
        print(f"  {m:18s}  {d}")
print(f"\nN_GRAPHS = {N_GRAPHS}, BATCH_SIZE = {BATCH_SIZE}")
print(f"Exp-2 results will be written to: {DRIVE_EXP_RESULTS}")

## 2. GPU check

In [ ]:
!nvidia-smi | head -20

## 3. Mount Drive

In [ ]:
from google.colab import drive
import os, shutil

MOUNTPOINT = "/content/drive"
# Colab errors if MOUNTPOINT already has files (re-run order, failed mount, etc.)
try:
    drive.flush_and_unmount()
except Exception:
    pass
if os.path.islink(MOUNTPOINT):
    os.remove(MOUNTPOINT)
elif os.path.isdir(MOUNTPOINT) and os.listdir(MOUNTPOINT):
    shutil.rmtree(MOUNTPOINT)
os.makedirs(MOUNTPOINT, exist_ok=True)
drive.mount(MOUNTPOINT)

## 4. Clone repo + install deps + symlink Drive folders

In [ ]:
import os, subprocess
%cd /content
if not os.path.isdir("gdl__2"):
    !git clone {GITHUB_REPO}
else:
    %cd gdl__2
    !git pull --ff-only || true
    %cd /content
%cd gdl__2
!bash run_colab/setup.sh

!rm -rf results_pcqm4m_subset datasets stability_baselines
!ln -sfn {DRIVE_RESULTS}    results_pcqm4m_subset
!ln -sfn {DRIVE_DATASETS}   datasets
!ln -sfn {DRIVE_STABILITY}  stability_baselines
!mkdir -p ANALYSIS/perturbation
!rm -rf ANALYSIS/perturbation/exp2_results
!ln -sfn {DRIVE_EXP_RESULTS} ANALYSIS/perturbation/exp2_results

!ls -la results_pcqm4m_subset datasets stability_baselines ANALYSIS/perturbation/exp2_results

## 5. Sanity checks — dataset + checkpoints exist

In [ ]:
from pathlib import Path
import glob

proc = Path(DRIVE_DATASETS) / "pcqm4m-v2" / "processed"
assert proc.is_dir(), f"Missing {proc}. Run 00_setup.ipynb first."
big = [p for p in proc.glob("*.pt") if p.stat().st_size > 500_000_000]
assert big, "No large processed .pt found in MyDrive/datasets/pcqm4m-v2/processed/"
for p in big:
    print(f"OK dataset  {p.name}  {p.stat().st_size / 1e9:.1f} GB")

print("\nCheckpoint discovery:")
missing = []
for method, dirs in METHOD_RUN_DIRS.items():
    for rd in dirs:
        cfg = sorted(glob.glob(os.path.join(rd, "**", "config.yaml"), recursive=True))
        ckpts = sorted(
            glob.glob(os.path.join(rd, "**", "*.ckpt"), recursive=True) +
            glob.glob(os.path.join(rd, "**", "*.pt"),   recursive=True))
        ckpts = [p for p in ckpts if "config" not in os.path.basename(p).lower()]
        if cfg and ckpts:
            print(f"  [OK]   {method:18s} {rd}  ->  cfg={cfg[-1]}  ckpt={ckpts[-1]}")
        else:
            print(f"  [MISS] {method:18s} {rd}  cfg={cfg}  ckpt={ckpts}")
            missing.append((method, rd))
if missing:
    raise FileNotFoundError(
        f"{len(missing)} run dir(s) missing config.yaml or a checkpoint. "
        "Upload them under MyDrive/stability_baselines/<method>/<seedX>/ and rerun."
    )

## 6. Run the experiment

The first call does a one-time candidate scan (all 73,545 test graphs, searching for near-degenerate pairs; ~5 min on CPU) and caches the result to `exp2_results/candidates.npz`. Subsequent runs reuse the cache. Already-completed `(method, seed)` result JSONs are skipped, so it is safe to interrupt and re-run.

In [ ]:
import subprocess, datetime, shlex
from pathlib import Path

SCRIPT = "ANALYSIS/perturbation/pe_stability_exp2.py"
OUT_DIR = Path("ANALYSIS/perturbation/exp2_results")

for method, dirs in METHOD_RUN_DIRS.items():
    for rd in dirs:
        seed_tag = Path(rd).name
        out_json = OUT_DIR / f"{method}__{seed_tag}.json"
        if out_json.exists():
            print(f"[skip] {method} :: {rd} — result already exists at {out_json}")
            continue
        cmd = ["python", SCRIPT,
               "--method", method,
               "--run-dirs", rd,
               "--n-graphs", str(N_GRAPHS),
               "--device", "cuda"]
        print(f"\n{'='*72}")
        print(f">>> {method}  {rd}")
        print(f">>> start: {datetime.datetime.now().isoformat(timespec='seconds')}")
        print(f">>> cmd:   {' '.join(shlex.quote(c) for c in cmd)}")
        print(f"{'='*72}", flush=True)
        ret = subprocess.call(cmd)
        status = "OK" if ret == 0 else f"FAILED (exit {ret})"
        print(f">>> done:  {datetime.datetime.now().isoformat(timespec='seconds')}  [{status}]")
        if ret != 0:
            raise RuntimeError(f"Exp2 failed for {method} :: {rd}")
print("\n=== All Exp 2 runs complete ===")

## 7. Aggregate + plots

Generates `exp2_bar.png` (exact-degeneracy bar chart) and `exp2_scatter.png` (per-graph std vs pair-gap).

In [ ]:
!python ANALYSIS/perturbation/pe_stability_exp2.py --plot-only

In [ ]:
from IPython.display import Image, display
import json
from pathlib import Path

out = Path("ANALYSIS/perturbation/exp2_results")
for name in ("exp2_bar.png", "exp2_scatter.png"):
    p = out / name
    if p.exists():
        print(f"--- {name} ---")
        display(Image(filename=str(p)))
    else:
        print(f"Plot not found: {p}")

summary_path = out / "exp2_summary.json"
if summary_path.exists():
    print("\n--- exp2_summary.json ---")
    print(json.dumps(json.load(open(summary_path)), indent=2))

## 8. Verify results on Drive

In [ ]:
!ls -la {DRIVE_EXP_RESULTS}
!du -sh {DRIVE_EXP_RESULTS}